<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 8.3 — RAG with ChromaDB (Persistent Vector Store)

In this notebook, you will build a simple Retrieval-Augmented Generation (RAG) workflow using:

- **MiniLM embeddings** (local)
- **ChromaDB** (persistent vector store)
- **Ollama** (local LLM)

### Learning goals
1) Store note embeddings persistently (so you don’t rebuild every time)
2) Retrieve notes relevant to a dementia-related query
3) Use retrieved notes to extract a **dementia phenotype (Yes/No)** with evidence

> Important: We do **not** pass gold labels into retrieval or prompting. Gold labels are used only later for evaluation.

## 1. Prepare Notes Data for Embedding

We will load the cleaned notes dataset produced in Lab 6:

- `lab6_clean_notes_baseline.parquet`

This dataset is already:
- restricted to the chosen 2-year window
- deduplicated across note versions
- reduced to the most recent note per (patient, visit, note type)

In [ ]:
# -----------------------------------------------------------
# 1.1. Load Cleaned Notes from Lab 6 (Parquet)
# -----------------------------------------------------------

from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

filepath = Path("data_files")

notes_df = pd.read_parquet(filepath / "lab6_clean_notes_baseline.parquet")

print("Notes loaded:", notes_df.shape)
display(notes_df.head(10))

## 2. Embed and Store Entire Clinical Notes in ChromaDB

In this step, we process and store each clinical note as a **complete document** in a ChromaDB vector store. This approach preserves full patient context, making it ideal for semantic search and downstream clinical reasoning.

### Step 2.1: Embed and Store Notes
1. **Embed Full Notes**
   Each note is converted into a semantic vector using a lightweight transformer model (`MiniLM`).

2. **Store in ChromaDB**
   The embedded vector and its associated metadata (e.g., patient ID, encounter number, visit date) are stored together in a persistent ChromaDB directory.

### Why This Approach?

Storing full notes is especially useful when:
- You want to retrieve the complete narrative for clinical context
- Your downstream task (e.g., summarization or decision support) depends on comprehensive input
- Each note fits within the input limit of a single LLM call

This setup supports more faithful summarization and reasoning than chunk-based approaches when notes are relatively short and self-contained.

<img src="./images/rag_full.png" alt="RAG Full" width="900">


In [ ]:
# -----------------------------------------------------------
# 2.1. Embed Notes Using MiniLM and Store in Persistent ChromaDB
# -----------------------------------------------------------

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Initialize embedding model (local)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

PERSIST_DIR = "./data_files/chroma_db_notes_epi"

# Create/load Chroma store
vectorstore = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embedding_model
)

# NOTE: If you re-run this cell, you may add duplicates.
# For the workshop: we will only add texts if the store is empty.
existing_count = vectorstore._collection.count()
print("Existing vectors in Chroma:", existing_count)

if existing_count == 0:
    documents = notes_df["note_txt_deid"].fillna("").tolist()
    metadata = notes_df[[
        "patient_num",
        "visit_id",
        "note_csn_id",
        "inpatient_note_type_dsc"
    ]].to_dict(orient="records")

    vectorstore.add_texts(texts=documents, metadatas=metadata)
    print(f"✅ Embedded and stored {len(documents)} notes in ChromaDB.")
else:
    print("✅ ChromaDB already populated. Skipping add_texts to avoid duplicates.")

## 3. Retrieving Clinical Notes with Similarity Score (RAG Retrieval with ChromaDB)

In this section, we retrieve relevant clinical notes from a ChromaDB vector store using vector similarity techniques. We explore both standard and advanced retrieval strategies to improve the relevance and diversity of retrieved context for downstream LLM generation.

### Key Retrieval Strategies

1. **Define a Clinical Query (Step 3.1)**
   - The user provides a natural language question (e.g., "Who has asthma and is taking Fluticasone and Albuterol?").

2. **Similarity Search with Scores (Step 3.2)**
   - Retrieves the top-k notes ranked by semantic similarity to the query.
   - Returns cosine-based relevance scores for each match.

3. **Score Threshold Filtering (Step 3.3)**
   - Filters out low-confidence matches using a minimum similarity score.
   - Retains only notes that meet a defined relevance threshold (e.g., ≥ 0.6).

4. **Maximal Marginal Relevance (MMR) Search (Step 3.4)**
   - Balances similarity and diversity.
   - Reduces redundancy by selecting a diverse subset of highly relevant notes.

### Why Use These Strategies?

High-quality retrieval is critical to the success of RAG workflows. These techniques help:
- Improve the contextual relevance of inputs to the LLM
- Filter out irrelevant or low-confidence documents
- Encourage diverse results to reduce bias and improve robustness

<img src="./images/rag_retrieval.png" alt="RAG Retrieval" width="900">


In [ ]:
# -----------------------------------------------------------
# 3.1. Define the Query for Clinical Note Retrieval
# -----------------------------------------------------------
# This cell defines a natural language query to search the embedded clinical notes
# stored in ChromaDB using semantic similarity.

# Concept:
# The query will be *** automatically embedded *** by the vectorstore before performing
# a semantic comparison against stored note vectors.
# -----------------------------------------------------------

# Example Clinical Query:
query = "Does this patient have a documented diagnosis of dementia or Alzheimer disease?"

# Display the query for reference
display(Markdown(f"### 🔍 Query: `{query}`"))


In [ ]:
# -----------------------------------------------------------
# 3.2. Perform Similarity Search with Relevance Scores
# -----------------------------------------------------------
# This cell retrieves the top-k clinical notes most semantically similar to the input query.
# Each result includes a cosine similarity score returned by LangChain's Chroma vectorstore wrapper.

# Function used:
# - vectorstore.similarity_search_with_relevance_scores(query, k=10)
#   Returns a list of (Document, score) tuples.

# Relevance Score Interpretation (heuristic only):
# - 0.90 – 1.00 → Highly relevant
# - 0.70 – 0.90 → Strong relevance
# - 0.50 – 0.70 → Moderate relevance
# - 0.30 – 0.50 → Low relevance
# - 0.00 – 0.30 → Minimal or no relevance
# -----------------------------------------------------------

# Perform the similarity search
results = vectorstore.similarity_search_with_relevance_scores(query, k=10)

# Display the results
display(Markdown("### 🔍 Retrieved Notes with Relevance Scores"))

for idx, (doc, score) in enumerate(results, 1):
    meta = doc.metadata or {}
    excerpt = (doc.page_content or "")[:900].replace("\n", " ")

    display(Markdown(
        f"---\n**Result {idx}**  \n"
        f"- **Relevance Score:** `{score:.4f}`  \n"
        f"- **patient_num:** `{meta.get('patient_num','N/A')}`  \n"
        f"- **visit_id:** `{meta.get('visit_id','N/A')}`  \n"
        f"- **note_csn_id:** `{meta.get('note_csn_id','N/A')}`  \n"
        f"- **note_type:** `{meta.get('inpatient_note_type_dsc','N/A')}`  \n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))


In [ ]:
# -----------------------------------------------------------
# 3.3. Use a Retriever with a Similarity Score Threshold
# -----------------------------------------------------------
# This cell configures a retriever that only returns documents whose
# cosine similarity scores exceed a predefined threshold.

# Parameters:
# - search_type="similarity_score_threshold": Enables threshold-based filtering.
# - search_kwargs={"k": 10, "score_threshold": 0.6}
#     - k: Maximum number of documents to *evaluate* (not necessarily return)
#     - score_threshold: Minimum similarity score (0–1) required for inclusion

# Purpose:
# Improves retrieval precision by excluding weak semantic matches—
# especially important for clinical reasoning and safety-critical applications.
# -----------------------------------------------------------

# Define threshold and top-k limit
# -----------------------------------------------------------
# 3.3. Retriever with Similarity Score Threshold
# -----------------------------------------------------------

score_threshold = 0.47
top_k = 10

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": top_k, "score_threshold": score_threshold}
)

threshold_docs = retriever.invoke(query)

display(Markdown(f"### 🔎 Retrieved Notes (Score ≥ {score_threshold})"))

for idx, doc in enumerate(threshold_docs, 1):
    meta = doc.metadata or {}
    excerpt = (doc.page_content or "")[:900].replace("\n", " ")

    display(Markdown(
        f"---\n**Doc {idx}**  \n"
        f"- **patient_num:** `{meta.get('patient_num','N/A')}`  \n"
        f"- **visit_id:** `{meta.get('visit_id','N/A')}`  \n"
        f"- **note_csn_id:** `{meta.get('note_csn_id','N/A')}`  \n"
        f"- **note_type:** `{meta.get('inpatient_note_type_dsc','N/A')}`  \n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))

display(Markdown(f"**✅ Total returned:** `{len(threshold_docs)}`"))


In [ ]:
# -----------------------------------------------------------
# 3.4. Perform Maximal Marginal Relevance (MMR) Search
# -----------------------------------------------------------
# This cell retrieves clinical notes using Maximal Marginal Relevance (MMR),
# an approach that balances query relevance with result diversity.

# Parameters:
# - fetch_k = 500: Number of top-ranked candidates to evaluate before reranking
# - k = 5: Final number of diverse, relevant documents to return
# - lambda_mult:
#     - 0.0 → prioritize diversity (minimal redundancy)
#     - 1.0 → prioritize similarity to the query
#     - 0.5 → balanced trade-off between diversity and relevance

# Purpose:
# MMR reduces redundancy by penalizing near-duplicate documents, while still
# prioritizing clinical notes that are highly relevant to the query.
# This is especially useful in settings where multiple perspectives or
# treatment variations are valuable (e.g., different asthma management plans).
# -----------------------------------------------------------

# Perform MMR search
results = vectorstore.max_marginal_relevance_search(
    query=query,
    k=5,
    fetch_k=500,
    lambda_mult=0.5
)

# Display results
display(Markdown("### 📘 Retrieved Clinical Notes Using Maximal Marginal Relevance (MMR)"))

for idx, doc in enumerate(results, 1):
    patient = doc.metadata.get("patient_num", "N/A")
    date = doc.metadata.get("start_date", "N/A")
    doc_id = getattr(doc, "id", "N/A")
    excerpt = doc.page_content[:1000].replace("\n", " ")

    display(Markdown(
        f"**Document {idx}**  \n"
        f"- **Patient Num:** `{patient}`  \n"
        f"- **Document ID:** `{doc_id}`  \n\n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))

display(Markdown(f"**✅ Total MMR Results Returned:** `{len(results)}`"))


## 4. Generating Structured Responses with an LLM (RAG Retrieval)

In this final step, we use a Large Language Model (LLM) to analyze the clinical notes retrieved in the previous section. By injecting these relevant notes into a structured prompt, we enable the LLM to generate clinically useful, structured responses.

This completes the Retrieval-Augmented Generation (RAG) pipeline, connecting search results to intelligent language generation.

### Key Steps

1. **Create a Prompt Template (Step 4.1)**
   - Defines a reusable prompt structure that includes placeholders for both the query and the retrieved clinical context.
   - Specifies a structured output format, including metadata fields such as patient ID, encounter date, and a clinical summary.

2. **Invoke the LLM with Retrieved Context (Step 4.2)**
   - Fills the prompt with retrieved notes and the clinical query.
   - Sends the prompt to a local LLM (e.g., Qwen2 or LLaMA 3 via Ollama).
   - Returns a structured, context-aware response to the clinical question.

### Why It Matters

This generation step demonstrates the power of combining semantic search with generative AI:
- Produces context-rich answers grounded in patient data
- Supports clinical summarization and decision support
- Enables zero-shot reasoning over real-world clinical notes

<img src="./images/rag_generation.png" alt="RAG Generation" width="1250">


In [ ]:
# -----------------------------------------------------------
# 4.1. Prompt Template: Dementia Phenotype from Retrieved Notes
# -----------------------------------------------------------
# This chat prompt guides the LLM to generate structured clinical summaries
# from notes retrieved via similarity search.

# Placeholders:
# - {retrieved_docs}: Injects top-matching clinical notes
# - {query}: A user-defined clinical question

# Output Expectations:
# - One structured summary per patient
# - Metadata included for traceability
# - Response based only on the most recent note per patient
# -----------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a clinical phenotyping assistant. "
     "Use ONLY the provided notes. Do NOT infer or invent. "
     "Do NOT mention privacy/redaction/date shifting/removed identifiers."),
    ("human",
     "Retrieved notes:\n\n{retrieved_docs}\n\n"
     "Clinical question: {query}\n\n"
     "Return a structured answer using bullet points:\n"
     "1) Dementia Phenotype (Yes/No)\n"
     "2) Evidence: up to 3 short direct quotes/phrases from the notes (or 'None')\n"
     "3) Rationale: 1–2 sentences grounded in the evidence\n"
     "4) Confidence: low/medium/high\n\n"
     "Rules for Dementia Phenotype:\n"
     "- Answer 'Yes' ONLY if dementia is explicitly documented (e.g., 'dementia', 'Alzheimer', "
     "'vascular dementia', 'Lewy body dementia') or clearly stated as a prior diagnosis.\n"
     "- If dementia is not explicitly mentioned, answer 'No' (do not infer from memory complaints alone)."
    )
])

print("✅ Prompt created and ready to use.")


In [ ]:
# -----------------------------------------------------------
# 4.2. Invoke LLM Using ALL Threshold-Retrieved Notes (with Traceability)
# -----------------------------------------------------------

from langchain_ollama import ChatOllama
from IPython.display import display, Markdown

MAX_CHARS_PER_NOTE = 2500

docs_for_llm = threshold_docs  # use threshold retrieval results

if len(docs_for_llm) == 0:
    display(Markdown("### No notes passed the threshold — nothing to send to the LLM."))
else:
    retrieved_blocks = []
    for rank, doc in enumerate(docs_for_llm, start=1):
        meta = doc.metadata or {}
        text = (doc.page_content or "").strip()[:MAX_CHARS_PER_NOTE]

        retrieved_blocks.append(
            f"---\n"
            f"[Retrieved Note #{rank}]\n"
            f"patient_num={meta.get('patient_num','N/A')} | visit_id={meta.get('visit_id','N/A')} | note_csn_id={meta.get('note_csn_id','N/A')}\n"
            f"note_type={meta.get('inpatient_note_type_dsc','N/A')} | create_dts_shifted={meta.get('create_dts_shifted','N/A')}\n\n"
            f"{text}\n"
        )

    retrieved_context = "\n".join(retrieved_blocks)

    display(Markdown(f"### Retrieved notes sent to LLM: {len(docs_for_llm)}"))
    display(Markdown(retrieved_context[:3000] + ("\n\n*(preview truncated)*" if len(retrieved_context) > 3000 else "")))

    final_prompt = prompt_template.format(
        retrieved_docs=retrieved_context,
        query=query
    )

    model = ChatOllama(model="llama4:scout", temperature=0.0)
    response = model.invoke(final_prompt)

    display(Markdown("### 🧠 LLM Output (Dementia Phenotype)"))
    display(Markdown(response.content))

In [ ]:
# -----------------------------------------------------------
# 4.3 (Bonus). Patient-Level Dementia Phenotyping from Notes
# -----------------------------------------------------------
# Features:
# - Processes patients sequentially
# - Early stop when dementia detected
# - Progress logging
# - Optional patient limit for faster runs
# -----------------------------------------------------------

import pandas as pd
from pathlib import Path
from langchain_ollama import ChatOllama

# Load evaluation labels (for later merge)
eval_df = pd.read_csv(Path("data_files") / "lab6_structured_dementia_eval.csv")

# LLM model
model = ChatOllama(model="qwen2", temperature=0.0)

# -----------------------------------------------------------
# OPTIONAL: limit number of patients (set to None to run all)
# -----------------------------------------------------------

PATIENT_LIMIT = 5   # change to None to process all patients

unique_patients = notes_df["patient_num"].unique().tolist()

if PATIENT_LIMIT:
    unique_patients = unique_patients[:PATIENT_LIMIT]

total_patients = len(unique_patients)

print(f"Patients to process: {total_patients}\n")

results = []

for idx, patient_num in enumerate(unique_patients, start=1):

    print(f"Processing patient {idx} of {total_patients} -> {patient_num}")

    patient_notes = notes_df[notes_df["patient_num"] == patient_num]

    dementia_found = False
    notes_checked = 0
    first_yes_note_csn = None
    first_yes_visit_id = None
    evidence_preview = None

    for _, row in patient_notes.iterrows():

        notes_checked += 1

        note_text = (row["note_txt_deid"] or "").strip()
        if len(note_text) == 0:
            continue

        retrieved_docs = (
            f"patient_num={row.get('patient_num','N/A')} | visit_id={row.get('visit_id','N/A')} | "
            f"note_csn_id={row.get('note_csn_id','N/A')}\n"
            f"note_type={row.get('inpatient_note_type_dsc','N/A')} | "
            f"create_dts_shifted={row.get('create_dts_shifted','N/A')}\n\n"
            f"{note_text}"
        )

        query = "Does the patient have a documented diagnosis of dementia or Alzheimer disease?"

        final_prompt = prompt_template.format(
            retrieved_docs=retrieved_docs,
            query=query
        )

        response = model.invoke(final_prompt).content

        # simple detection
        if "Dementia Phenotype" in response and "Yes" in response.split("Dementia Phenotype", 1)[1][:60]:

            dementia_found = True
            first_yes_note_csn = row.get("note_csn_id")
            first_yes_visit_id = row.get("visit_id")
            evidence_preview = response[:400]

            break

    print(f"   dementia: {'YES' if dementia_found else 'NO'}")
    print(f"   notes evaluated: {notes_checked}\n")

    results.append({
        "patient_num": patient_num,
        "rag_dementia": dementia_found,
        "notes_checked": notes_checked,
        "first_yes_note_csn_id": first_yes_note_csn,
        "first_yes_visit_id": first_yes_visit_id,
        "rag_output_preview": evidence_preview
    })

rag_df = pd.DataFrame(results)

print("RAG results shape:", rag_df.shape)
display(rag_df.head(10))

In [ ]:
# -----------------------------------------------------------
# 4.4 (Bonus). Merge RAG phenotype with structured + gold labels
# -----------------------------------------------------------

merged = eval_df.merge(rag_df, on="patient_num", how="left")

print("Merged shape:", merged.shape)
display(merged.head(10))

# Simple cross-tab vs gold
conf = pd.crosstab(
    merged["rag_dementia"].astype(bool),
    merged["gold_dementia"].astype(bool),
    rownames=["RAG Dementia"],
    colnames=["Gold Dementia"]
)
display(conf)